In [4]:
!pip install boto3 -q
!pip install xgboost -q


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from imblearn.combine import SMOTEENN
import boto3
from google.colab import userdata
import os
from io import BytesIO
from sklearn.ensemble import RandomForestClassifier
import sys
sys.path.append("/content/sample_data")
from lib.read_file import read_from_s3
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [6]:
try:
    aws_access_key = userdata.get('AWS_ACCESS_KEY_ID').strip()
    aws_secret_key = userdata.get('AWS_SECRET_ACCESS_KEY').strip()
    print("AWS Credentials loaded from Colab Secrets")
except:
    print("Please add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY in Colab Secrets first!")
    raise

os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_key
print("AWS credentials loaded successfully")

AWS_REGION = 'us-east-1'
s3 = boto3.client('s3',region_name = AWS_REGION)

# read from s3

found_key = None
response = s3.list_objects_v2(Bucket="ibm-internship-major-staging", Prefix="ibm_internship_major/staging/")
if 'Contents' in response:
    print("Files found inside the folder:\n")
    latest_date = response['Contents'][0]['LastModified']
    for obj in response['Contents']:
        latest_date = max(latest_date,obj['LastModified'])
        if latest_date == obj['LastModified']:
          found_key = obj["Key"]

else:
    print("No files found in this folder.")

if found_key != None:
  data = read_from_s3(
      bucket_name="ibm-internship-major-staging",
      s3_key= found_key
  )



AWS Credentials loaded from Colab Secrets
AWS credentials loaded successfully
Files found inside the folder:

File Found
Successfully loaded: s3://ibm-internship-major-staging/ibm_internship_major/staging/2026-07-27_14-53/IBM_Client_Churn_Analysis_Staging.csv
Shape: (7032, 51)


In [7]:
x = data.drop('Churn',axis=1)
x

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_1 - 12,tenure_group_13 - 24,tenure_group_25 - 36,tenure_group_37 - 48,tenure_group_49 - 60,tenure_group_61 - 72
0,0,29.85,29.85,True,False,False,True,True,False,True,...,False,False,True,False,True,False,False,False,False,False
1,0,56.95,1889.50,False,True,True,False,True,False,False,...,False,False,False,True,False,False,True,False,False,False
2,0,53.85,108.15,False,True,True,False,True,False,False,...,False,False,False,True,True,False,False,False,False,False
3,0,42.30,1840.75,False,True,True,False,True,False,True,...,True,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,True,False,True,False,True,False,False,...,False,False,True,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,0,84.80,1990.50,False,True,False,True,False,True,False,...,False,False,False,True,False,True,False,False,False,False
7028,0,103.20,7362.90,True,False,False,True,False,True,False,...,False,True,False,False,False,False,False,False,False,True
7029,0,29.60,346.45,True,False,False,True,False,True,True,...,False,False,True,False,True,False,False,False,False,False
7030,1,74.40,306.60,False,True,False,True,True,False,False,...,False,False,False,True,True,False,False,False,False,False


In [8]:
y = data['Churn']

In [9]:
y.value_counts()

,count
Churn,
0,5163
1,1869


# Decision Tree

In [10]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)
model_dt = DecisionTreeClassifier(criterion='gini',random_state=100,max_depth=6,min_samples_leaf=8)
model_dt.fit(x_train,y_train)
y_pred = model_dt.predict(x_test)
model_dt.score(x_test,y_test)
print(classification_report(y_test, y_pred, labels=[0,1]))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1014
           1       0.62      0.49      0.55       393

    accuracy                           0.77      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.76      0.77      0.77      1407



In [11]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [12]:
model_dt=DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=6, min_samples_leaf=8)

In [13]:
model_dt.fit(x_train,y_train)

DecisionTreeClassifier(max_depth=6, min_samples_leaf=8, random_state=100)

In [14]:
y_pred=model_dt.predict(x_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [15]:
model_dt.score(x_test,y_test)

0.7818052594171997

In [16]:
print(classification_report(y_test, y_pred, labels=[0,1]))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1031
           1       0.62      0.48      0.54       376

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407



As you can see that the accuracy is quite low, and as it's an imbalanced dataset, we shouldn't consider Accuracy as our metrics to measure the model, as Accuracy is cursed in imbalanced datasets.
Hence, we need to check recall, precision & f1 score for the minority class, and it's quite evident that the precision, recall & f1 score is too low for Class 1, i.e. churned customers.

In [17]:
sm = SMOTEENN()
X_resampled, y_resampled = sm.fit_resample(x,y)

In [18]:
xr_train,xr_test,yr_train,yr_test=train_test_split(X_resampled, y_resampled,test_size=0.2)

In [19]:
model_dt_smote=DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=6, min_samples_leaf=8)

In [20]:
model_dt_smote.fit(xr_train,yr_train)
yr_predict = model_dt_smote.predict(xr_test)
model_score_r = model_dt_smote.score(xr_test, yr_test)
print(model_score_r)
print(metrics.classification_report(yr_test, yr_predict))

0.9320305862361937
              precision    recall  f1-score   support

           0       0.94      0.91      0.93       543
           1       0.93      0.95      0.94       634

    accuracy                           0.93      1177
   macro avg       0.93      0.93      0.93      1177
weighted avg       0.93      0.93      0.93      1177



In [21]:

print(metrics.confusion_matrix(yr_test, yr_predict))

[[495  48]
 [ 32 602]]


In [22]:
model_rf=RandomForestClassifier(n_estimators=100, criterion='gini', random_state = 100,max_depth=6, min_samples_leaf=8)

In [23]:
model_rf.fit(x_train,y_train)

RandomForestClassifier(max_depth=6, min_samples_leaf=8, random_state=100)

In [24]:
y_pred=model_rf.predict(x_test)

In [25]:
model_rf.score(x_test,y_test)

0.8002842928216063

In [26]:
print(classification_report(y_test, y_pred, labels=[0,1]))

              precision    recall  f1-score   support

           0       0.83      0.92      0.87      1031
           1       0.69      0.47      0.55       376

    accuracy                           0.80      1407
   macro avg       0.76      0.69      0.71      1407
weighted avg       0.79      0.80      0.79      1407



In [27]:
sm = SMOTEENN()
X_resampled1, y_resampled1 = sm.fit_resample(x,y)

In [28]:
xr_train1,xr_test1,yr_train1,yr_test1=train_test_split(X_resampled1, y_resampled1,test_size=0.2)

In [29]:
model_rf_smote=RandomForestClassifier(n_estimators=100, criterion='gini', random_state = 100,max_depth=6, min_samples_leaf=8)

In [30]:
model_rf_smote.fit(xr_train1,yr_train1)

RandomForestClassifier(max_depth=6, min_samples_leaf=8, random_state=100)

In [31]:
yr_predict1 = model_rf_smote.predict(xr_test1)

In [32]:
model_score_r1 = model_rf_smote.score(xr_test1, yr_test1)

In [33]:
print(model_score_r1)
print(metrics.classification_report(yr_test1, yr_predict1))

0.9314140558848434
              precision    recall  f1-score   support

           0       0.95      0.90      0.92       549
           1       0.92      0.96      0.94       632

    accuracy                           0.93      1181
   macro avg       0.93      0.93      0.93      1181
weighted avg       0.93      0.93      0.93      1181



In [34]:
print(metrics.confusion_matrix(yr_test1, yr_predict1))

[[493  56]
 [ 25 607]]


In [35]:
from sklearn.decomposition import PCA
pca = PCA(0.9)
xr_train_pca = pca.fit_transform(xr_train1)
xr_test_pca = pca.transform(xr_test1)
explained_variance = pca.explained_variance_ratio_

In [36]:
model=RandomForestClassifier(n_estimators=100, criterion='gini', random_state = 100,max_depth=6, min_samples_leaf=8)
model.fit(xr_train_pca,yr_train1)
yr_predict_pca = model.predict(xr_test_pca)
model_score_r_pca = model.score(xr_test_pca, yr_test1)
print(model_score_r_pca)
print(metrics.classification_report(yr_test1, yr_predict_pca))

0.707874682472481
              precision    recall  f1-score   support

           0       0.70      0.64      0.67       549
           1       0.71      0.77      0.74       632

    accuracy                           0.71      1181
   macro avg       0.71      0.70      0.70      1181
weighted avg       0.71      0.71      0.71      1181



In [37]:
model_nb = GaussianNB()
model_nb.fit(xr_train1, yr_train1)

yr_pred_nb = model_nb.predict(xr_test1)

print("Naive Bayes Accuracy:", model_nb.score(xr_test1, yr_test1))
print(metrics.classification_report(yr_test1, yr_pred_nb))
print(metrics.confusion_matrix(yr_test1, yr_pred_nb))

Naive Bayes Accuracy: 0.8882303132938189
              precision    recall  f1-score   support

           0       0.90      0.85      0.88       549
           1       0.88      0.92      0.90       632

    accuracy                           0.89      1181
   macro avg       0.89      0.89      0.89      1181
weighted avg       0.89      0.89      0.89      1181

[[467  82]
 [ 50 582]]


In [38]:
model_lr = LogisticRegression(max_iter=1000, random_state=100)
model_lr.fit(xr_train1, yr_train1)

yr_pred_lr = model_lr.predict(xr_test1)

print("Logistic Regression Accuracy:", model_lr.score(xr_test1, yr_test1))
print(metrics.classification_report(yr_test1, yr_pred_lr))
print(metrics.confusion_matrix(yr_test1, yr_pred_lr))

Logistic Regression Accuracy: 0.9424216765453006
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       549
           1       0.95      0.95      0.95       632

    accuracy                           0.94      1181
   macro avg       0.94      0.94      0.94      1181
weighted avg       0.94      0.94      0.94      1181

[[515  34]
 [ 34 598]]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [39]:
model_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=100,
    eval_metric='logloss'
)

model_xgb.fit(xr_train1, yr_train1)

yr_pred_xgb = model_xgb.predict(xr_test1)

print("XGBoost Accuracy:", model_xgb.score(xr_test1, yr_test1))
print(metrics.classification_report(yr_test1, yr_pred_xgb))
print(metrics.confusion_matrix(yr_test1, yr_pred_xgb))

XGBoost Accuracy: 0.9652836579170194
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       549
           1       0.96      0.97      0.97       632

    accuracy                           0.97      1181
   macro avg       0.97      0.96      0.97      1181
weighted avg       0.97      0.97      0.97      1181

[[524  25]
 [ 16 616]]


In [40]:
param_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [4, 6, 8, 10, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None]
}

rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=100),
    param_distributions=param_rf,
    n_iter=30,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=100
)

rf_random.fit(xr_train1, yr_train1)
print("Best RF Parameters:", rf_random.best_params_)
print("Best RF Score:", rf_random.best_score_)

best_rf = rf_random.best_estimator_
print(classification_report(yr_test1, best_rf.predict(xr_test1)))

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best RF Parameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best RF Score: 0.9589141660479163
              precision    recall  f1-score   support

           0       0.96      0.93      0.95       549
           1       0.94      0.97      0.96       632

    accuracy                           0.95      1181
   macro avg       0.95      0.95      0.95      1181
weighted avg       0.95      0.95      0.95      1181



In [41]:
param_xgb = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 3, 5]
}

xgb_random = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=100, eval_metric='logloss'),
    param_distributions=param_xgb,
    n_iter=40,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=100
)

xgb_random.fit(xr_train1, yr_train1)

print("Best XGBoost Parameters:", xgb_random.best_params_)
print("Best XGBoost Score:", xgb_random.best_score_)

best_xgb = xgb_random.best_estimator_
print(classification_report(yr_test1, best_xgb.predict(xr_test1)))



Fitting 3 folds for each of 40 candidates, totalling 120 fits
Best XGBoost Parameters: {'subsample': 0.6, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.6}
Best XGBoost Score: 0.9599932164728938
              precision    recall  f1-score   support

           0       0.96      0.95      0.95       549
           1       0.95      0.97      0.96       632

    accuracy                           0.96      1181
   macro avg       0.96      0.96      0.96      1181
weighted avg       0.96      0.96      0.96      1181



In [44]:
import pickle

# Save all models
pickle.dump(model_rf_smote, open('model_rf_smote.sav', 'wb'))
pickle.dump(model_nb, open('model_nb.sav', 'wb'))
pickle.dump(model_lr, open('model_lr.sav', 'wb'))
pickle.dump(model_xgb, open('model_xgb.sav', 'wb'))
pickle.dump(best_rf, open('best_rf.sav', 'wb'))
pickle.dump(best_xgb, open('best_xgb.sav', 'wb'))

print("All models saved successfully!")

load_model = pickle.load(open('model_xgb.sav', 'rb'))
print("XGBoost loaded successfully. Score:", load_model.score(xr_test1, yr_test1))

All models saved successfully!
XGBoost loaded successfully. Score: 0.9652836579170194


Our final model i.e. RF Classifier with SMOTEENN, is now ready and dumped in model.sav, which we will use and prepare API's so that we can access our model from UI.

In [43]:
# !pip install boto3 -q
# import boto3
# import pandas as pd
# from io import BytesIO
# AWS_REGION = 'us-east-1'

# def read_from_s3(bucket_name, s3_key):
#     try:
#         s3 = boto3.client('s3',region_name = AWS_REGION)
#         response = s3.get_object(Bucket=bucket_name, Key=s3_key)
#         print("File Found")
#         file_content = response['Body'].read()

#         if s3_key.lower().endswith('.xlsx'):
#             df = pd.read_excel(BytesIO(file_content))

#         elif s3_key.lower().endswith('.csv'):
#             df = pd.read_csv(BytesIO(file_content))
#         else:
#             df = pd.read_excel(BytesIO(file_content))  # default

#         print(f"Successfully loaded: s3://{bucket_name}/{s3_key}")
#         print(f"Shape: {df.shape}")
#         return df

#     except Exception as e:
#         print(f"Error: {e}")
#         return None
